## Agent-to-Agent Conversation

In [1]:
from agents import (
    Agent,
    set_default_openai_api,
    function_tool,
    Runner,
    SQLiteSession,
)
import os
import nest_asyncio
from dotenv import load_dotenv
import requests

nest_asyncio.apply()
load_dotenv()
set_default_openai_api("chat_completions")  # use chat completions api
set_default_openai_api(os.getenv("OPENAI_API_KEY"))

### Define customer agent

In [33]:
customer_agent = Agent(
    name="Customer Agent",
    instructions=(
        "You are a customer ONLY. "
        "You must NOT give financial advice, analysis, or recommendations. "
        "You must ONLY:\n"
        "- Answer questions asked by the financial planner\n"
        "- Express preferences, concerns, or constraints\n"
        "- Ask for clarification if confused\n\n"
        "If the planner asks a question, you MUST answer it directly.\n"
        "Do NOT introduce new advice or planning concepts."
    ),
    model="gpt-4.1-mini",
)

### Define financial agent

In [34]:
financial_agent = Agent(
    name="Financial Planning Agent",
    instructions=(
        "You are a professional financial planner. "
        "Use the full conversation history to provide consistent advice."
    ),
    model="gpt-4.1-mini",
)

### Orchestrate the Conversation (with History)

In [35]:
async def run_conversation_simulation(
    initial_customer_message: str, num_turns: int = 5
):
    turn = 1
    conversation_history = []
    conversation_history.append({"role": "user", "content": initial_customer_message})

    while turn < num_turns:
        # Financial planner responds to customer input
        planner_result = await Runner.run(
            financial_agent,
            input=conversation_history,
        )

        planner_reply = planner_result.final_output

        conversation_history.append({"role": "assistant", "content": planner_reply})

        # Customer responds to financial planner's advice
        customer_result = await Runner.run(
            customer_agent,
            input=conversation_history,
        )
        customer_reply = customer_result.final_output

        conversation_history.append({"role": "user", "content": customer_reply})
        turn += 1
    return conversation_history

In [36]:
sim_history = await run_conversation_simulation(
    initial_customer_message="I earn $80,000 a year, have $20,000 in savings, and want to buy a house in five years. What should I think about?",
    num_turns=3,
)
sim_history

[{'role': 'user',
  'content': 'I earn $80,000 a year, have $20,000 in savings, and want to buy a house in five years. What should I think about?'},
 {'role': 'assistant',
  'content': 'With an $80,000 annual income, $20,000 in savings, and a goal to buy a house in five years, here’s what you should consider to prepare financially:\n\n1. **Determine Your Home Budget:**\n   - Research home prices in your desired area to get a realistic idea of what you can afford.\n   - A common rule is to aim for a home price about 3–4 times your annual income, so roughly $240,000 to $320,000, but this varies by location.\n\n2. **Save for the Down Payment:**\n   - Most lenders require 3% to 20% down payment.\n   - If aiming for 20% down to avoid Private Mortgage Insurance (PMI), on a $300,000 home, that means $60,000.\n   - You already have $20,000 saved, so you would want to save an additional $40,000 over 5 years, or $8,000 per year (~$667/month).\n\n3. **Improve/Review Your Credit Score:**\n   - A h